# 05b — Comprehensive evaluation and comparison

Generate requested ablation plots, metrics, interpretability analysis, and summary tables.


## Install dependencies


In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torch_geometric', 'numpy', 'pandas', 'matplotlib', 'scikit-learn', 'scipy', 'seaborn', 'umap-learn'])


0

## Setup


In [2]:
from pathlib import Path
import os, sys, json, time, random
import numpy as np
import pandas as pd

USE_DRIVE = True
SEED = 42

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/BulkCellGNN_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, '/content/drive/MyDrive/BulkCellGNN_data')

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)

print('DATA_DIR =', DATA_DIR)

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from scipy.stats import ttest_ind
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import bulkcell_gnn as bcg
import bulkcell_gnn_shilu as bcg_shilu

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


Mounted at /content/drive
DATA_DIR = /content/drive/MyDrive/BulkCellGNN_data
device = cuda


## Load artifacts


In [3]:
logs = json.loads((DATA_DIR / 'training_logs_ablation.json').read_text(encoding='utf-8'))

bulk_x = torch.tensor(np.load(DATA_DIR / 'bulk_expr_sym.npy'), dtype=torch.float32).to(device)
cell_x = torch.tensor(np.load(DATA_DIR / 'cell_expr_full.npy'), dtype=torch.float32).to(device)
labels = torch.tensor(np.load(DATA_DIR / 'bulk_labels.npy'), dtype=torch.long).to(device)
cell_types = torch.tensor(np.load(DATA_DIR / 'cell_types_full.npy'), dtype=torch.long).to(device)
patient_ids = np.load(DATA_DIR / 'cell_patient_ids_full.npy', allow_pickle=True)

edge_BB = torch.load(DATA_DIR / 'edge_BB_full.pt', map_location=device)
edge_CC = torch.load(DATA_DIR / 'edge_CC_full.pt', map_location=device)
edge_BC = torch.load(DATA_DIR / 'edge_BC_full.pt', map_location=device)

type_names = json.loads((DATA_DIR / 'cell_type_names_full.json').read_text(encoding='utf-8'))
val_mask = torch.tensor(np.load(DATA_DIR / 'val_mask_full.npy'), dtype=torch.bool).to(device)


## Build/load three models


In [4]:
def make_model(which):
    if which == 'shilu':
        m = bcg_shilu.BulkCellGNN(bulk_x.shape[1], cell_x.shape[1], d_latent=256, n_classes=2, n_cell_types=len(type_names), n_layers=2, dropout=0.3, cell_type_names=type_names)
        m.load_state_dict(torch.load(DATA_DIR / 'model_best_shilu.pt', map_location=device))
    elif which == 'gelu':
        m = bcg.BulkCellGNN(bulk_x.shape[1], cell_x.shape[1], d_latent=256, n_classes=2, n_cell_types=len(type_names), n_layers=2, dropout=0.3, cell_type_names=type_names)
        m.load_state_dict(torch.load(DATA_DIR / 'model_best_gelu.pt', map_location=device))
    else:
        import torch.nn as nn
        m = bcg.BulkCellGNN(bulk_x.shape[1], cell_x.shape[1], d_latent=256, n_classes=2, n_cell_types=len(type_names), n_layers=2, dropout=0.3, cell_type_names=type_names)
        def repl(mod):
            for n, c in mod.named_children():
                if isinstance(c, nn.GELU):
                    setattr(mod, n, nn.ReLU())
                else:
                    repl(c)
        repl(m)
        m.load_state_dict(torch.load(DATA_DIR / 'model_best_relu.pt', map_location=device))
    return m.to(device).eval()

models = {k: make_model(k) for k in ['shilu', 'gelu', 'relu']}


## Compute metrics and gamma arrays


In [5]:
results = {}
for key, m in models.items():
    with torch.no_grad():
        out = m(bulk_x, cell_x, edge_BB, edge_CC, edge_BC, cell_types, return_gamma=True)
    probs = F.softmax(out['logits'][val_mask], dim=-1)[:, 1].detach().cpu().numpy()
    y_true = labels[val_mask].detach().cpu().numpy()
    y_pred = (probs >= 0.5).astype(int)
    results[key] = {
        'out': out,
        'probs': probs,
        'y_true': y_true,
        'auc': float(roc_auc_score(y_true, probs)),
        'acc': float(accuracy_score(y_true, y_pred)),
        'prec': float(precision_score(y_true, y_pred, zero_division=0)),
        'rec': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'cm': confusion_matrix(y_true, y_pred),
        'gamma': out['gamma'].detach().cpu().numpy(),
    }
print({k: v['auc'] for k, v in results.items()})


{'shilu': 0.998641304347826, 'gelu': 0.9952445652173914, 'relu': 0.9972826086956521}


## Section 1: training overlays + ShiLU parameter evolution


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for key, label in [('shilu', 'ShiLU-SwishTanh'), ('gelu', 'GELU'), ('relu', 'ReLU')]:
    lg = logs[key]
    axes[0, 0].plot(lg['epoch'], lg['loss_total'], label=label)
    axes[0, 1].plot(lg['epoch'], lg['loss_cls'], label=label)
    axes[1, 0].plot(lg['epoch'], lg['val_auc'], label=label)
axes[1, 0].axhline(0.964, linestyle='--', color='gray', label='SVM baseline 0.964')
axes[0, 0].set_title('Total loss')
axes[0, 1].set_title('Classification loss')
axes[1, 0].set_title('Validation AUC')
for ax in axes.flat[:3]:
    ax.set_xlabel('Epoch')
    ax.legend()

for n, vals in logs['shilu'].get('shilu_params', {}).items():
    if n.endswith('alpha'):
        axes[1, 1].plot(range(1, len(vals) + 1), vals, label=n)
axes[1, 1].set_title('ShiLU alpha evolution')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend(fontsize=7)
fig.tight_layout()
fig.savefig(DATA_DIR / 'cmp_section1_training_curves.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)


## Section 2: ROC, metric bars, confusion matrices


In [7]:
fig, ax = plt.subplots(figsize=(6, 6))
for key, label, color in [('shilu', 'ShiLU-SwishTanh', '#2563EB'), ('gelu', 'GELU', '#059669'), ('relu', 'ReLU', '#DC2626')]:
    fpr, tpr, _ = roc_curve(results[key]['y_true'], results[key]['probs'])
    ax.plot(fpr, tpr, label=f'{label} (AUC={results[key]["auc"]:.3f})', color=color)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.legend(); ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC comparison')
fig.savefig(DATA_DIR / 'cmp_section2_roc.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

metrics_df = pd.DataFrame([
    {'Model': 'ShiLU-SwishTanh', 'AUC': results['shilu']['auc'], 'Accuracy': results['shilu']['acc'], 'Precision': results['shilu']['prec'], 'Recall': results['shilu']['rec'], 'F1': results['shilu']['f1']},
    {'Model': 'GELU', 'AUC': results['gelu']['auc'], 'Accuracy': results['gelu']['acc'], 'Precision': results['gelu']['prec'], 'Recall': results['gelu']['rec'], 'F1': results['gelu']['f1']},
    {'Model': 'ReLU', 'AUC': results['relu']['auc'], 'Accuracy': results['relu']['acc'], 'Precision': results['relu']['prec'], 'Recall': results['relu']['rec'], 'F1': results['relu']['f1']},
])
metrics_df.to_csv(DATA_DIR / 'ablation_metrics.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 4))
metrics_long = metrics_df.melt(id_vars=['Model'], var_name='Metric', value_name='Score')
sns.barplot(data=metrics_long, x='Metric', y='Score', hue='Model', ax=ax)
ax.set_ylim(0, 1.05)
fig.savefig(DATA_DIR / 'cmp_section2_metrics_bar.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (key, label) in zip(axes, [('shilu', 'ShiLU-SwishTanh'), ('gelu', 'GELU'), ('relu', 'ReLU')]):
    sns.heatmap(results[key]['cm'], annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(label)
    ax.set_xlabel('Pred')
    ax.set_ylabel('True')
fig.tight_layout()
fig.savefig(DATA_DIR / 'cmp_section2_confusion_matrices.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)


## Section 3: gamma heatmaps, profile bar chart, and t-tests


In [8]:
def gamma_heatmap_for(ax, gamma, labels_np, title):
    msi = gamma[labels_np == 1].mean(axis=0)
    mss = gamma[labels_np == 0].mean(axis=0)
    mat = np.vstack([msi, mss])
    sns.heatmap(mat, annot=True, fmt='.3f', cmap='mako', yticklabels=['MSI', 'MSS'], xticklabels=type_names, ax=ax)
    ax.set_title(title)

labels_np = labels.detach().cpu().numpy()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
gamma_heatmap_for(axes[0], results['shilu']['gamma'], labels_np, 'ShiLU gamma')
gamma_heatmap_for(axes[1], results['gelu']['gamma'], labels_np, 'GELU gamma')
gamma_heatmap_for(axes[2], results['relu']['gamma'], labels_np, 'ReLU gamma')
fig.tight_layout()
fig.savefig(DATA_DIR / 'cmp_section3_gamma_heatmaps.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

bar_rows = []
for key, label in [('shilu', 'ShiLU-SwishTanh'), ('gelu', 'GELU'), ('relu', 'ReLU')]:
    gm = results[key]['gamma']
    for i, t in enumerate(type_names):
        bar_rows.append({'Model': label, 'CellType': t, 'GammaMean': float(gm[:, i].mean())})
bar_df = pd.DataFrame(bar_rows)
fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=bar_df, x='CellType', y='GammaMean', hue='Model', ax=ax)
fig.savefig(DATA_DIR / 'cmp_section3_gamma_profile_bar.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

t_rows = []
for key, label in [('shilu', 'ShiLU-SwishTanh'), ('gelu', 'GELU'), ('relu', 'ReLU')]:
    gm = results[key]['gamma']
    for i, t in enumerate(type_names):
        msi_vals = gm[labels_np == 1, i]
        mss_vals = gm[labels_np == 0, i]
        _, p = ttest_ind(msi_vals, mss_vals, equal_var=False)
        t_rows.append({'Model': label, 'CellType': t, 'p_value': float(p)})
pd.DataFrame(t_rows).to_csv(DATA_DIR / 'gamma_ttests.csv', index=False)


## Section 4: embedding UMAP comparisons


In [9]:
hB = {k: torch.load(DATA_DIR / f'hB_{k}.pt', map_location='cpu').numpy() for k in ['shilu', 'gelu', 'relu']}
hC = {k: torch.load(DATA_DIR / f'hC_{k}.pt', map_location='cpu').numpy() for k in ['shilu', 'gelu', 'relu']}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (k, label) in zip(axes, [('shilu', 'ShiLU-SwishTanh'), ('gelu', 'GELU'), ('relu', 'ReLU')]):
    emb = umap.UMAP(n_components=2, random_state=SEED, min_dist=0.2).fit_transform(hC[k])
    sc = ax.scatter(emb[:, 0], emb[:, 1], s=2, c=cell_types.detach().cpu().numpy(), cmap='tab10', alpha=0.7)
    ax.set_title(f'Cell UMAP - {label}')
fig.colorbar(sc, ax=axes, fraction=0.02, pad=0.01)
fig.savefig(DATA_DIR / 'cmp_section4_cell_umap_triptych.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (k, label) in zip(axes, [('shilu', 'ShiLU-SwishTanh'), ('gelu', 'GELU'), ('relu', 'ReLU')]):
    emb = umap.UMAP(n_components=2, random_state=SEED, min_dist=0.2).fit_transform(hB[k])
    ax.scatter(emb[:, 0], emb[:, 1], s=25, c=labels_np, cmap='coolwarm', alpha=0.85)
    ax.set_title(f'Bulk UMAP - {label}')
fig.savefig(DATA_DIR / 'cmp_section4_bulk_umap_triptych.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

best_key = max(['shilu', 'gelu', 'relu'], key=lambda k: results[k]['auc'])
best_label = {'shilu': 'ShiLU-SwishTanh', 'gelu': 'GELU', 'relu': 'ReLU'}[best_key]

emb_best_cell = umap.UMAP(n_components=2, random_state=SEED, min_dist=0.2).fit_transform(hC[best_key])
fig, ax = plt.subplots(figsize=(7, 5))
sc_best = ax.scatter(emb_best_cell[:, 0], emb_best_cell[:, 1], s=2, c=cell_types.detach().cpu().numpy(), cmap='tab10', alpha=0.7)
ax.set_title(f'Best model cell UMAP ({best_label})')
fig.colorbar(sc_best, ax=ax, fraction=0.046, pad=0.04)
fig.savefig(DATA_DIR / 'best_model_cell_umap.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

emb_best_bulk = umap.UMAP(n_components=2, random_state=SEED, min_dist=0.2).fit_transform(hB[best_key])
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(emb_best_bulk[:, 0], emb_best_bulk[:, 1], s=25, c=labels_np, cmap='coolwarm', alpha=0.85)
ax.set_title(f'Best model bulk UMAP ({best_label})')
fig.savefig(DATA_DIR / 'best_model_bulk_umap.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

emb_patient = umap.UMAP(n_components=2, random_state=SEED, min_dist=0.2).fit_transform(hC['shilu'])
unique_pat = pd.factorize(patient_ids)[0]
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(emb_patient[:, 0], emb_patient[:, 1], s=2, c=unique_pat, cmap='tab20', alpha=0.7)
ax.set_title('Cell UMAP by patient ID (ShiLU)')
fig.savefig(DATA_DIR / 'cmp_section4_cell_umap_by_patient.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

(DATA_DIR / 'best_model_key.txt').write_text(best_key, encoding='utf-8')


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr

5

## Section 5: ShiLU-specific analysis


In [10]:
shp = logs['shilu'].get('shilu_params', {})
layer_names = sorted(shp.keys())
final_vals = [shp[n][-1] for n in layer_names]

fig, ax = plt.subplots(figsize=(10, max(3, len(layer_names) * 0.25)))
sns.heatmap(np.array(final_vals).reshape(-1, 1), annot=True, fmt='.3f', cmap='viridis', yticklabels=layer_names, xticklabels=['final'], ax=ax)
fig.savefig(DATA_DIR / 'cmp_section5_shilu_final_param_heatmap.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

x = np.linspace(-4, 4, 400)
alpha_vals = [v for n, v in shp.items() if n.endswith('alpha')]
gamma_vals = [v for n, v in shp.items() if n.endswith('gamma')]
delta_vals = [v for n, v in shp.items() if n.endswith('delta')]
a = float(np.mean([vals[-1] for vals in alpha_vals])) if alpha_vals else 0.5
g = float(np.mean([vals[-1] for vals in gamma_vals])) if gamma_vals else 1.0
d = float(np.mean([vals[-1] for vals in delta_vals])) if delta_vals else 0.0
shilu = a * (x * (1 / (1 + np.exp(-g * (x + d))))) + (1 - a) * np.tanh(x)
gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * np.power(x, 3))))
relu = np.maximum(0, x)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, shilu, label=f'ShiLU (a={a:.3f}, g={g:.3f}, d={d:.3f})')
ax.plot(x, gelu, label='GELU')
ax.plot(x, relu, label='ReLU')
ax.legend(); ax.set_title('Activation shape comparison')
fig.savefig(DATA_DIR / 'cmp_section5_activation_shapes.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
for n, vals in shp.items():
    if n.endswith('alpha'):
        ax.plot(range(1, len(vals) + 1), vals, label=n)
ax.axhline(0.5, linestyle='--', color='gray', alpha=0.6, label='alpha init')
ax.set_title('ShiLU alpha convergence')
ax.legend(fontsize=7)
fig.savefig(DATA_DIR / 'cmp_section5_alpha_convergence.png', dpi=200, facecolor='white', bbox_inches='tight')
plt.close(fig)


## Section 6: summary table


In [11]:
summary_df = pd.DataFrame([
    {'Model': 'ShiLU-SwishTanh', 'Val AUC': results['shilu']['auc'], 'Best Epoch': logs['shilu']['best_epoch'], 'Params': sum(p.numel() for p in models['shilu'].parameters()), 'Training time': logs['shilu']['training_time_sec']},
    {'Model': 'GELU', 'Val AUC': results['gelu']['auc'], 'Best Epoch': logs['gelu']['best_epoch'], 'Params': sum(p.numel() for p in models['gelu'].parameters()), 'Training time': logs['gelu']['training_time_sec']},
    {'Model': 'ReLU', 'Val AUC': results['relu']['auc'], 'Best Epoch': logs['relu']['best_epoch'], 'Params': sum(p.numel() for p in models['relu'].parameters()), 'Training time': logs['relu']['training_time_sec']},
])
summary_df.to_csv(DATA_DIR / 'model_summary.csv', index=False)
print(summary_df.to_markdown(index=False))


| Model           |   Val AUC |   Best Epoch |   Params |   Training time |
|:----------------|----------:|-------------:|---------:|----------------:|
| ShiLU-SwishTanh |  0.998641 |           21 | 20696358 |         30.1351 |
| GELU            |  0.995245 |            6 | 20696340 |         27.1888 |
| ReLU            |  0.997283 |           10 | 20696340 |         27.2317 |


## Outcome narrative


In [12]:
auc_vals = {k: results[k]['auc'] for k in ['shilu', 'gelu', 'relu']}
best = max(auc_vals, key=auc_vals.get)
spread = max(auc_vals.values()) - min(auc_vals.values())
if best == 'shilu' and (auc_vals['shilu'] - max(auc_vals['gelu'], auc_vals['relu']) > 0.005):
    outcome = 'A) ShiLU-SwishTanh improves AUC — supports domain-motivated activation design.'
elif spread <= 0.01:
    outcome = 'B) Performance is similar — ShiLU-SwishTanh matches SOTA without degrading.'
else:
    outcome = 'C) ShiLU-SwishTanh has different gamma profile — interpretation differs even if AUC is similar.'
print(outcome)
(DATA_DIR / 'ablation_outcome.txt').write_text(outcome, encoding='utf-8')


B) Performance is similar — ShiLU-SwishTanh matches SOTA without degrading.


75

## Final verification


In [13]:
expected = ['ablation_metrics.csv','gamma_ttests.csv','model_summary.csv','cmp_section1_training_curves.png','cmp_section2_roc.png','cmp_section2_metrics_bar.png','cmp_section2_confusion_matrices.png','cmp_section3_gamma_heatmaps.png','cmp_section3_gamma_profile_bar.png','cmp_section4_cell_umap_triptych.png','cmp_section4_bulk_umap_triptych.png','best_model_cell_umap.png','best_model_bulk_umap.png','best_model_key.txt','cmp_section4_cell_umap_by_patient.png','cmp_section5_shilu_final_param_heatmap.png','cmp_section5_activation_shapes.png','cmp_section5_alpha_convergence.png','ablation_outcome.txt']
missing = []
for name in expected:
    p = DATA_DIR / name
    if p.exists():
        print('OK ', name)
    else:
        print('MISSING', name)
        missing.append(name)
if missing:
    raise FileNotFoundError('Missing outputs: ' + ', '.join(missing))
print('Notebook 05b complete')


OK  ablation_metrics.csv
OK  gamma_ttests.csv
OK  model_summary.csv
OK  cmp_section1_training_curves.png
OK  cmp_section2_roc.png
OK  cmp_section2_metrics_bar.png
OK  cmp_section2_confusion_matrices.png
OK  cmp_section3_gamma_heatmaps.png
OK  cmp_section3_gamma_profile_bar.png
OK  cmp_section4_cell_umap_triptych.png
OK  cmp_section4_bulk_umap_triptych.png
OK  best_model_cell_umap.png
OK  best_model_bulk_umap.png
OK  best_model_key.txt
OK  cmp_section4_cell_umap_by_patient.png
OK  cmp_section5_shilu_final_param_heatmap.png
OK  cmp_section5_activation_shapes.png
OK  cmp_section5_alpha_convergence.png
OK  ablation_outcome.txt
Notebook 05b complete
